# Zone Timelapses — Precomputation

Builds one compact parquet per **zone × year** so the frontend can play a
whole-year timelapse from a single ~5–15 MB download, with zero per-frame
network and free multi-flag filtering.

Each zone has a fixed center / zoom / resolution, so its data is fully
precomputable. Output layout:
```
data/parquet/zones/<zone-id>-<YEAR>.parquet
```
Schema: `date, lat, lon, flag, fishing_hours, illegal_hours`

`illegal_hours` is fishing inside a foreign EEZ — baked in here via the
`eez_grid` join so the frontend never has to join at runtime.


## Config


In [1]:
from pathlib import Path
import os, time
import duckdb

# --- Locate the repo root (works whether the notebook runs from / or /notebooks) ---
ROOT = Path.cwd()
while not (ROOT / 'data' / 'parquet').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / 'data' / 'parquet').exists(), 'Could not locate data/parquet'

DAILY_DIR = ROOT / 'data' / 'parquet' / 'daily_split'
EEZ_GRID  = ROOT / 'data' / 'parquet' / 'eez_grid.parquet'
OUT_DIR   = ROOT / 'data' / 'parquet' / 'zones'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Year to build (single build-time choice; matches TIMELAPSE_YEAR on the frontend) ---
YEAR = 2023

# --- Bbox padding around each zone center, in degrees. Kept large on purpose:
#     the frontend viewport is clipped from this, so it must exceed any
#     plausible on-screen extent at the zone's fixed zoom. ---
BBOX_PAD_LON = 25.0
BBOX_PAD_LAT = 18.0

# --- Zones — mirrors frontend/src/data/zones.ts (id, lon, lat, zoom). Keep in sync. ---
ZONES = [
    {'id': 'south-china-sea', 'lon': 114.0, 'lat': 14.0, 'zoom': 4.0},
    {'id': 'grand-banks',     'lon': -52.0, 'lat': 46.5, 'zoom': 5.0},
    {'id': 'west-africa',     'lon': -17.0, 'lat': 12.0, 'zoom': 4.5},
    {'id': 'bering-sea',      'lon': -174.0,'lat': 59.0, 'zoom': 4.0},
    {'id': 'north-sea',       'lon': 3.0,   'lat': 56.5, 'zoom': 4.0},
]

def zoom_to_resolution(zoom: float) -> float:
    """Mirror of zoomToResolution() in frontend/src/utils.ts."""
    if zoom <= 2: return 0.512
    if zoom <= 4: return 0.32
    if zoom <= 5: return 0.16
    if zoom <= 6: return 0.08
    if zoom <= 7: return 0.04
    if zoom <= 8: return 0.02
    return 0.01

print('repo root :', ROOT)
print('daily dir :', DAILY_DIR, '-', len(list(DAILY_DIR.glob(f'fleet-daily-{YEAR}-*.parquet'))), 'files for', YEAR)
print('output dir:', OUT_DIR)


repo root : /Users/plouc314/Documents/epfl/MA3/viz/a-fishing-story
daily dir : /Users/plouc314/Documents/epfl/MA3/viz/a-fishing-story/data/parquet/daily_split - 365 files for 2023
output dir: /Users/plouc314/Documents/epfl/MA3/viz/a-fishing-story/data/parquet/zones


## Build

For each zone: aggregate every daily file of `YEAR` over the padded bbox to the
zone's display resolution, grouped by `(date, lat, lon, flag)`. The `eez_grid`
LEFT JOIN (at the native 0.1° grid, before aggregation) yields `illegal_hours`.


In [2]:
def build_zone(zone: dict, year: int) -> Path:
    res = zoom_to_resolution(zone['zoom'])
    lon_min, lon_max = zone['lon'] - BBOX_PAD_LON, zone['lon'] + BBOX_PAD_LON
    lat_min, lat_max = zone['lat'] - BBOX_PAD_LAT, zone['lat'] + BBOX_PAD_LAT
    daily_glob = str(DAILY_DIR / f'fleet-daily-{year}-*.parquet')
    out_path = OUT_DIR / f"{zone['id']}-{year}.parquet"

    con = duckdb.connect()
    con.execute(f"""
        COPY (
            SELECT
                regexp_extract(d.filename, '\\d{{4}}-\\d{{2}}-\\d{{2}}') AS date,
                (ROUND(d.cell_ll_lat / {res}) * {res})::FLOAT AS lat,
                (ROUND(d.cell_ll_lon / {res}) * {res})::FLOAT AS lon,
                d.flag AS flag,
                SUM(d.fishing_hours)::FLOAT AS fishing_hours,
                SUM(CASE WHEN e.eez_iso IS NOT NULL
                          AND e.eez_iso != d.flag
                          AND e.eez_iso != 'INT'
                         THEN d.fishing_hours ELSE 0 END)::FLOAT AS illegal_hours
            FROM read_parquet('{daily_glob}', filename = true) d
            LEFT JOIN read_parquet('{EEZ_GRID}') e
              ON ROUND(d.cell_ll_lat::DOUBLE * 10)::INTEGER = ROUND(e.cell_ll_lat * 10)::INTEGER
             AND ROUND(d.cell_ll_lon::DOUBLE * 10)::INTEGER = ROUND(e.cell_ll_lon * 10)::INTEGER
            WHERE d.cell_ll_lat BETWEEN {lat_min} AND {lat_max}
              AND d.cell_ll_lon BETWEEN {lon_min} AND {lon_max}
              AND d.flag IS NOT NULL
            GROUP BY 1, 2, 3, 4
            HAVING SUM(d.fishing_hours) > 0
        ) TO '{out_path}' (FORMAT parquet, COMPRESSION zstd)
    """)
    con.close()
    return out_path

for zone in ZONES:
    t = time.time()
    path = build_zone(zone, YEAR)
    size_mb = path.stat().st_size / 1e6
    print(f"{zone['id']:<18} res={zoom_to_resolution(zone['zoom']):<5} "
          f"{size_mb:6.2f} MB  ({time.time() - t:4.1f}s)")


south-china-sea    res=0.32    6.15 MB  ( 1.8s)
grand-banks        res=0.16    1.78 MB  ( 1.4s)
west-africa        res=0.16    2.99 MB  ( 0.9s)
bering-sea         res=0.32    0.31 MB  ( 0.5s)
north-sea          res=0.32    5.77 MB  ( 1.1s)


## Verify


In [3]:
con = duckdb.connect()
for zone in ZONES:
    path = OUT_DIR / f"{zone['id']}-{YEAR}.parquet"
    row = con.sql(f"""
        SELECT COUNT(*) AS rows,
               COUNT(DISTINCT date) AS days,
               COUNT(DISTINCT flag) AS flags,
               ROUND(SUM(fishing_hours)) AS hours,
               ROUND(100.0 * SUM(illegal_hours) / SUM(fishing_hours), 1) AS illegal_pct
        FROM '{path}'
    """).fetchone()
    print(f"{zone['id']:<18} rows={row[0]:>8}  days={row[1]:>3}  flags={row[2]:>4}  "
          f"hours={row[3]:>12,.0f}  illegal={row[4]}%")
con.close()


south-china-sea    rows=  735084  days=365  flags= 202  hours=  47,612,181  illegal=18.8%
grand-banks        rows=  235901  days=365  flags=  23  hours=   1,259,436  illegal=2.9%
west-africa        rows=  350149  days=365  flags=  58  hours=   2,104,137  illegal=37.0%
bering-sea         rows=   44008  days=365  flags=  12  hours=     445,418  illegal=2.8%
north-sea          rows=  709665  days=365  flags=  54  hours=  10,161,306  illegal=22.4%


## Upload to HuggingFace

Mirrors `hf.py`. Uploads each zone-year file under the same `parquet/` repo path
the frontend reads from.


In [ ]:
from huggingface_hub import HfApi

api = HfApi()
for zone in ZONES:
    path = OUT_DIR / f"{zone['id']}-{YEAR}.parquet"
    repo_path = str(path.relative_to(ROOT / 'data'))
    api.upload_file(
        path_or_fileobj=str(path),
        path_in_repo=repo_path,
        repo_id='Plouc314/fishing',
        repo_type='dataset',
    )
    print('uploaded', repo_path)
